# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR^2](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the [mlcroissant](https://mlcommons.github.io/croissant/python/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: 

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install the required mlcroissant library (if not already installed)
!pip install --quiet mlcroissant

## 1. Data Loading
Load dataset metadata and explore key descriptors using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print a summary
print("Title      :", dataset.metadata.name)
print("Version    :", dataset.metadata.version)
print("Identifier :", dataset.metadata.identifier)
print("Published  :", dataset.metadata.datePublished)
print("License    :", dataset.metadata.license)
print("\nDescription:")
print(dataset.metadata.description)


## 2. Data Overview
Review available record sets, their field `@id`s, and data structure.

A record set is a core data table (analogous to a DataFrame or a CSV table) defined by the dataset Croissant schema. Each record set is uniquely identified by its `@id`.

In [ ]:
# List record sets and their field (column) @ids
from pprint import pprint

# Get all record set IDs
record_sets_ids = [rs['@id'] for rs in dataset.metadata.as_json_dict().get('recordSet', [])]

print(f"Record Sets Found ({len(record_sets_ids)}): ")
pprint(record_sets_ids)

# For each record set, list its field @ids
for recset in dataset.metadata.as_json_dict().get('recordSet', []):
    print(f"\nRecord Set @id: {recset['@id']}")
    field_ids = [field['@id'] for field in recset.get('field', [])]
    print(f"Fields (@id): {field_ids}")


### Preview Records from Each Record Set
Print the first (few) records from a chosen record set.

_You should specify the `@id` of the record set to preview. We select the first in the list for demonstration._

In [ ]:
# Preview first five records from the first record set (if available)
if record_sets_ids:
    record_set_id = record_sets_ids[0]
    print(f"\nFirst five records from record set: {record_set_id}\n{'-'*60}")
    for i, record in enumerate(dataset.records(record_set=record_set_id)):
        pprint(record)
        if i >= 4:
            break

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis. All references use entity `@id`s for clarity and consistency.

In [ ]:
# Load all record sets into pandas DataFrames
dataframes = {}
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set @id: {record_set_id}")

# For demo, select the first non-empty record set
chosen_record_set_id = None
for rsid, df in dataframes.items():
    if not df.empty:
        chosen_record_set_id = rsid
        break

if chosen_record_set_id:
    print(f"\nColumns for record set {chosen_record_set_id}:\n{dataframes[chosen_record_set_id].columns.tolist()}")
    display(dataframes[chosen_record_set_id].head())
else:
    print("No record sets with data available.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records based on a numeric field, normalizing data, and grouping records. All field names referenced by their column (`@id`) in the Croissant schema.

> _For demonstration, we try to infer a numeric field and a group field from available columns._

In [ ]:
import numpy as np

if chosen_record_set_id:
    df = dataframes[chosen_record_set_id]

    # Try to find a numeric field (float or int)
    numeric_col_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_col_candidates:
        numeric_field = numeric_col_candidates[0]
        print(f"Using numeric field (column @id): {numeric_field}")
    else:
        print("No numeric fields found for EDA. Please check your dataset.")
        numeric_field = None

    # Filtering: Use median if proper threshold, else default to 0
    if numeric_field:
        median_val = df[numeric_field].median()
        threshold = median_val if np.isfinite(median_val) else 0

        # Filter for values above threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records where '{numeric_field}' > {threshold:.2f} (median): {len(filtered_df)} records")
        display(filtered_df.head())

        # Normalize field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to find a group field (i.e., categorical/text field)
        group_col_candidates = df.select_dtypes(include=[object, "category"]).columns.tolist()
        # Exclude the numeric field chosen
        group_col_candidates = [c for c in group_col_candidates if c != numeric_field]
        group_field = group_col_candidates[0] if group_col_candidates else None

        if group_field:
            print(f"Grouping by field (column @id): {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"mean_{numeric_field}")
            display(grouped_df.head())
        else:
            print("No group field available for grouping analysis.")
    else:
        print("Cannot perform EDA without numeric field.")
else:
    print("No record set available for EDA.")

## 5. Visualization
Visualize the distribution of the main numeric field and relationships with the group field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if chosen_record_set_id and numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

* In this notebook, we loaded the [FAIR^2 Mass Spectrometry](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library, explored record sets and fields, extracted tables as DataFrames, and performed basic EDA and visualizations.
* All dataset entities (record sets, columns) are referenced by their `@id` for clarity and reproducibility.
* You can build on this notebook to perform in-depth analysis of protein abundances and covariates, or to integrate the data into larger proteomics workflows.